# Utility Function at Different Train / Bootstrap Sizes

Tests the **PhysioNet 2019 Challenge utility function** (official implementation) 
across a grid of training-set sizes and bootstrap sample sizes.

### Utility Formula (Official PhysioNet 2019)

**Time windows** (relative to true sepsis onset):
- `dt_early = -12 h` — earliest beneficial prediction
- `dt_optimal = -6 h` — optimal prediction time
- `dt_late = +3 h` — latest beneficial prediction

**Utility scoring** (piecewise linear):

| Time Window | Prediction | Utility |
|---|---|---|
| -12 to -6 h | Alarm (TP) | Ramps from 0 → 1 |
| -6 to +3 h | Alarm (TP) | Decays from 1 → 0 |
| +3 h onwards | Any | -2 (too late) |
| Any time | Missed (FN) | -2 (worst case) |
| Any time | False alarm (FP) | -0.05 (mild penalty) |
| Any time | Correct (TN) | 0 (baseline) |

### Normalisation

Per-patient score:
```
score = (observed - inaction) / (best - inaction)
```

Then averaged across patients to prevent long-stay patients from dominating.

### Test Grid
| Parameter | Values |
|---|---|
| Training patients | 50 · 100 · 200 · 300 |
| Bootstrap patients / sample | 25 · 50 · 100 |
| Bootstrap iterations | 10 (set `N_ITER` higher for production) |
| Model | LogisticGLM |

**Reference:** PhysioNet 2019 Challenge official evaluate_sepsis_score.py

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm

# When executed via nbconvert, CWD is the notebook's own directory (rebuild/).
# Insert '.' so that rebuild/*.py modules are importable directly.
sys.path.insert(0, '.')
logging.basicConfig(level=logging.WARNING)   # suppress INFO noise in notebook

from config       import Config, TrainingConfig, BootstrapConfig
from data_loader  import (
    load_physionet_files,
    add_hours_until_sepsis,
    split_patients_by_status,
    get_rows_for_patients,
)
from bootstrap    import BootstrapResampler
from models       import LogisticGLM
from training     import BootstrapEvaluator

print('Imports OK')

## 1  Load and prepare data

The raw PhysioNet PSV files contain `ICULOS` (ICU hour counter, 1-indexed) and
`SepsisLabel` (0 → non-septic; 1 → sepsis active).

The official PhysioNet 2019 utility function uses **SepsisLabel directly**, not 
computed hours. The `add_hours_until_sepsis()` function is kept for reference and 
per-group analysis, but utility scores are computed from raw labels.

In [ ]:
# Notebook lives in rebuild/; data is one level up at ../data/physionet_sepsis
DATA_DIR = Path('../data/physionet_sepsis')

print('Loading PSV files …')
raw_df = load_physionet_files(DATA_DIR)
print(f'  {raw_df["patient_id"].nunique():,} patients · {len(raw_df):,} rows')

print('Computing hours_until_sepsis …')
df = add_hours_until_sepsis(raw_df, keep_post_onset=True)

# Quick sanity-check
n_septic   = df['hours_until_sepsis'].notna().sum()
n_at_onset = (df['hours_until_sepsis'] == 0).sum()
n_pre      = (df['hours_until_sepsis'] >  0).sum()
n_post     = (df['hours_until_sepsis'] <  0).sum()
print(f'  Septic rows        : {n_septic:,}')
print(f'    – pre-onset  (h > 0): {n_pre:,}')
print(f'    – at onset   (h = 0): {n_at_onset:,}')
print(f'    – post-onset (h < 0): {n_post:,}')
print(f'  Non-septic rows    : {df["hours_until_sepsis"].isna().sum():,}')

# Distribution of pre-onset lead times
pre_onset = df[df['hours_until_sepsis'] >= 0]['hours_until_sepsis']
print(f'\nPre-onset hours_until_sepsis — '
      f'median {pre_onset.median():.0f} h, '
      f'p25 {pre_onset.quantile(.25):.0f} h, '
      f'p75 {pre_onset.quantile(.75):.0f} h, '
      f'max {pre_onset.max():.0f} h')

## 2  PhysioNet 2019 utility function preview

Official formula: Piecewise linear utility with time windows relative to true sepsis onset.

The oracle (optimal strategy) alarms during the beneficial window [-12h, +3h].
Early predictions (>12h early) don't help; late predictions (+3h after onset) hurt.

In [ ]:
# The utility function is piecewise linear with 3 time windows:
# Segment 1: [-12h, -6h]  → linear ramp from 0 to 1
# Segment 2: [-6h, +3h]   → linear decay from 1 to 0  
# Segment 3: [+3h, ∞)     → flat at -2 (too late)

print("PhysioNet 2019 utility function:")
print("  Time window [-12h, -6h]: U increases from 0 to 1")
print("  Time window [-6h, +3h]:  U decreases from 1 to 0")
print("  Time window [+3h, ∞):    U = -2 (too late)")
print("  False alarm cost:        U_FP = -0.05")
print("  Missed sepsis cost:      U_FN = -2")

## 3  Experiment grid

In [ ]:
TRAIN_SIZES      = [50, 100, 200, 300]
BOOTSTRAP_SIZES  = [25, 50, 100]
N_ITER           = 10    # iterations per config; raise to 100+ for production
RANDOM_STATE     = 42

print(f'Grid  : {len(TRAIN_SIZES)} train sizes × {len(BOOTSTRAP_SIZES)} bootstrap sizes')
print(f'        = {len(TRAIN_SIZES)*len(BOOTSTRAP_SIZES)} configs × {N_ITER} iterations')
print(f'        = {len(TRAIN_SIZES)*len(BOOTSTRAP_SIZES)*N_ITER} total bootstrap evaluations')

## 4  Run experiments

In [ ]:
records = []   # one dict per (train_size, bootstrap_size, iteration)

pbar_outer = tqdm(TRAIN_SIZES, desc='Train size', position=0)

for train_size in pbar_outer:
    pbar_outer.set_postfix(n_train=train_size)

    # ── Patient split ─────────────────────────────────────────────────────────
    train_pids, boot_pids = split_patients_by_status(
        df, n_train_patients=train_size,
        random_state=RANDOM_STATE, stratify_by_sepsis=True,
    )
    train_df = get_rows_for_patients(df, train_pids)

    for boot_size in tqdm(BOOTSTRAP_SIZES, desc='  Boot size', position=1, leave=False):

        # ── Bootstrap resampler ───────────────────────────────────────────────
        resampler = BootstrapResampler(
            bootstrap_pool_patient_ids = boot_pids,
            full_df                    = df,
            n_iterations               = N_ITER,
            bootstrap_sample_size      = boot_size,
            random_state               = RANDOM_STATE,
        )

        # ── Fit model once on training set ────────────────────────────────────
        evaluator = BootstrapEvaluator(
            model            = LogisticGLM(),
            train_df         = train_df,
            label_column     = 'SepsisLabel',
            patient_id_column = 'patient_id',
        )

        # ── Evaluate on each bootstrap sample ─────────────────────────────────
        for i in range(N_ITER):
            _, boot_df = resampler.generate_iteration(i)
            m = evaluator.evaluate_iteration(
                boot_df, i,
                compute_per_group = True,
                group_column      = 'Gender',
            )

            row = dict(
                train_size    = train_size,
                boot_size     = boot_size,
                iteration     = i,
                n_samples     = m['n_samples'],
                n_positive    = m['n_positive'],
                prevalence    = m['prevalence'],
                auroc         = m.get('auroc',    np.nan),
                recall        = m.get('recall',   np.nan),
                accuracy      = m.get('accuracy', np.nan),
                f1            = m.get('f1',       np.nan),
                utility       = m.get('utility',  np.nan),
            )

            # per-group utility
            if 'per_group' in m:
                for g, gm in m['per_group'].items():
                    row[f'utility_g{int(g)}'] = gm.get('utility', np.nan)
                    row[f'auroc_g{int(g)}']   = gm.get('auroc',   np.nan)
                    row[f'recall_g{int(g)}']  = gm.get('recall',  np.nan)

            records.append(row)

results = pd.DataFrame(records)
print(f'\n✓  {len(results):,} rows collected')
results.head(3)

## 5  Aggregate per configuration

In [ ]:
metric_cols = ['auroc', 'recall', 'accuracy', 'f1', 'utility',
               'utility_g0', 'utility_g1', 'auroc_g0', 'auroc_g1',
               'recall_g0', 'recall_g1']
metric_cols = [c for c in metric_cols if c in results.columns]

agg = (
    results
    .groupby(['train_size', 'boot_size'])[metric_cols]
    .agg(['mean', 'std', 'median'])
    .round(4)
)

# Flatten multi-level columns
agg.columns = ['_'.join(c) for c in agg.columns]
agg = agg.reset_index()
print(agg[['train_size', 'boot_size',
           'utility_mean', 'utility_std',
           'auroc_mean',   'recall_mean']].to_string(index=False))

In [ ]:
# Compute utility fairness gap (Female − Male, Gender 0 vs 1)
if 'utility_g0' in results.columns and 'utility_g1' in results.columns:
    results['utility_gap'] = results['utility_g0'] - results['utility_g1']
    print('Utility gap (Female − Male) sample stats:')
    print(results.groupby('train_size')['utility_gap']
          .agg(['mean', 'std', 'median']).round(4))

## 6  Visualisations

### 6a  Utility & AUROC vs training size (one line per bootstrap size)

In [ ]:
PALETTE = {25: '#1f77b4', 50: '#ff7f0e', 100: '#2ca02c'}
METRICS  = [
    ('utility',  'Utility (PhysioNet guide)'),
    ('auroc',    'AUROC'),
    ('recall',   'Recall'),
    ('accuracy', 'Accuracy'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
fig.suptitle('Model performance vs training-set size', fontsize=14, fontweight='bold')

for ax, (metric, title) in zip(axes.flat, METRICS):
    sub = results.groupby(['train_size', 'boot_size'])[metric]
    for bsize in BOOTSTRAP_SIZES:
        grp = sub.agg(['mean', 'std']).reset_index()
        g   = grp[grp['boot_size'] == bsize]
        if g['mean'].isna().all():
            continue
        ax.errorbar(
            g['train_size'], g['mean'], yerr=g['std'],
            label=f'boot={bsize}', marker='o', capsize=4,
            color=PALETTE.get(bsize, 'gray'),
        )
    ax.set_title(title)
    ax.set_xlabel('Training patients')
    ax.set_ylabel(title)
    ax.set_xticks(TRAIN_SIZES)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('scaling_metrics.png', dpi=150)
plt.show()
print('✓  scaling_metrics.png')

### 6b  Heatmaps: training size × bootstrap size

In [ ]:
def _heatmap(metric, title, vmin=None, vmax=None, cmap='RdYlGn', ax=None):
    pivot = (
        results.groupby(['train_size', 'boot_size'])[metric]
        .mean().unstack('boot_size')
    )
    if pivot.isna().all(axis=None):
        if ax:
            ax.text(0.5, 0.5, f'{title}\nNo data', ha='center', va='center',
                    transform=ax.transAxes)
        return
    sns.heatmap(
        pivot, annot=True, fmt='.4f', cmap=cmap,
        vmin=vmin, vmax=vmax,
        cbar_kws={'label': metric},
        ax=ax,
    )
    if ax:
        ax.set_title(title)
        ax.set_xlabel('Bootstrap sample size')
        ax.set_ylabel('Training patients')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mean metric — training size × bootstrap size', fontsize=13)
_heatmap('utility', 'Utility',  cmap='RdYlGn', ax=axes[0])
_heatmap('auroc',   'AUROC',    cmap='RdYlGn', ax=axes[1])
plt.tight_layout()
plt.savefig('heatmap_utility_auroc.png', dpi=150)
plt.show()
print('✓  heatmap_utility_auroc.png')

### 6c  Utility distributions (box plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# By training size
if 'utility' in results.columns and results['utility'].notna().any():
    sns.boxplot(
        data=results, x='train_size', y='utility',
        palette='Set2', ax=axes[0],
    )
    axes[0].axhline(0, ls='--', color='red', lw=1.2, label='Inaction baseline')
    axes[0].set_title('Utility distribution by training size')
    axes[0].set_xlabel('Training patients')
    axes[0].set_ylabel('Utility (normalised)')
    axes[0].legend(fontsize=8)
    axes[0].grid(axis='y', alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No utility data', ha='center', va='center',
                 transform=axes[0].transAxes)

# By bootstrap size
if 'utility' in results.columns and results['utility'].notna().any():
    sns.boxplot(
        data=results, x='boot_size', y='utility',
        palette='Set3', ax=axes[1],
    )
    axes[1].axhline(0, ls='--', color='red', lw=1.2)
    axes[1].set_title('Utility distribution by bootstrap size')
    axes[1].set_xlabel('Bootstrap sample size')
    axes[1].set_ylabel('Utility (normalised)')
    axes[1].grid(axis='y', alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No utility data', ha='center', va='center',
                 transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('utility_boxplots.png', dpi=150)
plt.show()
print('✓  utility_boxplots.png')

### 6d  Utility fairness gap (Female − Male) by training size

In [ ]:
if 'utility_gap' in results.columns and results['utility_gap'].notna().any():
    fig, ax = plt.subplots(figsize=(8, 4))

    gap_summary = (
        results.groupby('train_size')['utility_gap']
        .agg(['mean', 'std']).reset_index()
    )
    ax.bar(
        gap_summary['train_size'].astype(str),
        gap_summary['mean'],
        yerr=gap_summary['std'],
        capsize=5,
        color=['#e06c75' if v < 0 else '#98c379' for v in gap_summary['mean']],
        edgecolor='black',
    )
    ax.axhline(0, color='black', lw=1)
    ax.set_xlabel('Training patients')
    ax.set_ylabel('Utility gap  (Female − Male)')
    ax.set_title('Utility fairness gap by training size\n'
                 '(green = female favoured  |  red = male favoured)')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('utility_fairness_gap.png', dpi=150)
    plt.show()
    print('✓  utility_fairness_gap.png')
else:
    print('No per-group utility data available — '
          'check that Gender column is present in bootstrap samples')

### 6e  Threshold sensitivity (one bootstrap sample, biggest training set)

In [ ]:
from utility import utility_curve, find_optimal_threshold
from training import extract_Xy

# Re-use the largest training set
train_pids_big, boot_pids_big = split_patients_by_status(
    df, n_train_patients=max(TRAIN_SIZES), random_state=RANDOM_STATE,
)
train_df_big = get_rows_for_patients(df, train_pids_big)

model_big = LogisticGLM()
X_tr, y_tr = extract_Xy(train_df_big, 'SepsisLabel')
model_big.fit(X_tr, y_tr)

# One bootstrap sample
res_big = BootstrapResampler(
    boot_pids_big, df, n_iterations=1,
    bootstrap_sample_size=max(BOOTSTRAP_SIZES), random_state=RANDOM_STATE,
)
_, sample_df = res_big.generate_iteration(0)

X_s, _   = extract_Xy(sample_df, 'SepsisLabel')
y_proba  = model_big.predict_proba(X_s)[:, 1]
labels   = sample_df['SepsisLabel'].values  # Use raw labels, not hours_until_sepsis
pids     = sample_df['patient_id'].values

curve_df = utility_curve(y_proba, labels, patient_ids=pids)
opt_thresh, opt_util = find_optimal_threshold(y_proba, labels, patient_ids=pids)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(curve_df['threshold'], curve_df['utility'],
        lw=2, color='steelblue', label='Utility')
ax.axvline(opt_thresh, ls='--', color='green', lw=1.5,
           label=f'Optimal threshold = {opt_thresh:.2f}  (utility = {opt_util:.3f})')
ax.axvline(0.5, ls=':', color='grey', lw=1.5, label='Default threshold = 0.5')
ax.axhline(0,   ls='--', color='red',  lw=1.2, label='Inaction baseline')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Normalised utility')
ax.set_title(f'Utility vs threshold  (training={max(TRAIN_SIZES)} patients,'
             f' bootstrap sample size={max(BOOTSTRAP_SIZES)})')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('utility_vs_threshold.png', dpi=150)
plt.show()
print(f'Optimal threshold: {opt_thresh:.2f}   Max utility: {opt_util:.4f}')

## 7  Summary table and export

In [ ]:
summary = (
    results
    .groupby('train_size')[['utility', 'auroc', 'recall', 'f1']]
    .agg(['mean', 'std'])
    .round(4)
)
summary.columns = ['_'.join(c) for c in summary.columns]
print('\nSummary — averaged across bootstrap sizes and iterations')
print(summary.to_string())

In [ ]:
results.to_csv('utility_scaling_results.csv', index=False)
summary.to_csv('utility_scaling_summary.csv')
print('✓  utility_scaling_results.csv')
print('✓  utility_scaling_summary.csv')

## 8  Key findings

In [ ]:
print('=' * 60)
print('KEY FINDINGS')
print('=' * 60)

# Best config by utility
if results['utility'].notna().any():
    best = agg.sort_values('utility_mean', ascending=False).iloc[0]
    print(f'\nBest utility config:')
    print(f'  Train size   : {best["train_size"]:.0f} patients')
    print(f'  Boot size    : {best["boot_size"]:.0f} patients / sample')
    print(f'  Utility mean : {best["utility_mean"]:.4f} ± {best["utility_std"]:.4f}')
    print(f'  AUROC mean   : {best["auroc_mean"]:.4f}')

    # Scaling trend
    trend = results.groupby('train_size')['utility'].mean()
    delta = trend.iloc[-1] - trend.iloc[0]
    pct   = delta / abs(trend.iloc[0]) * 100 if trend.iloc[0] != 0 else float('nan')
    print(f'\nScaling: utility {train_size}→{max(TRAIN_SIZES)} patients')
    print(f'  {trend.iloc[0]:.4f} → {trend.iloc[-1]:.4f}  ({pct:+.1f}%)')
else:
    print('\n⚠  hours_until_sepsis was all-NaN — '
          'check that add_hours_until_sepsis() ran on your DataFrame.')

# Fairness gap
if 'utility_gap' in results.columns and results['utility_gap'].notna().any():
    gap_by_train = results.groupby('train_size')['utility_gap'].mean().abs()
    print(f'\nMean |utility gap| by training size:')
    for ts, gap in gap_by_train.items():
        print(f'  {ts:3d} patients: {gap:.4f}')

print(f'\nOptimal threshold (largest config): {opt_thresh:.2f}')
print(f'Max utility at optimal threshold  : {opt_util:.4f}')
print(f'Utility at default threshold 0.5  : {curve_df.set_index("threshold").loc[0.50, "utility"]:.4f}')